# Graph-of-Thought — Colab GPU runner (T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arrafmousa/graph-of-thought/blob/master/notebooks/colab_run.ipynb)

Runs the **reasoning-graph POC** on GSM8K with a **frozen** `meta-llama/Llama-3.2-1B-Instruct`
(FP16), following `AGENTS.md` §31. Set **Runtime → Change runtime type → T4 GPU** first.

Two phases:
1. **Tuning** (`scripts/tune_graph.py`) — sweep merge heuristics × thresholds over 25
   questions; emits one dashboard per configuration + a comparison dashboard. Pick a
   heuristic + threshold.
2. **Full run** (`scripts/generate_graphs.py`) — generate graphs with the selected
   heuristic/threshold and render the per-question graph report.

`Llama-3.2` is **gated**: request access on its model page and add a Hugging Face token
as a Colab secret named `HF_TOKEN`. Dataset + model are chosen in the config files
(explicit HF ids/revisions). Each run writes `output/<run_id>/`; the last cell zips it.


In [ ]:
# 1) Clone the repo
# Public repo:
!git clone https://github.com/arrafmousa/graph-of-thought.git

# Private repo instead? Store a GitHub token as a Colab secret named GH_TOKEN, then:
# from google.colab import userdata
# tok = userdata.get('GH_TOKEN')
# !git clone https://{tok}@github.com/arrafmousa/graph-of-thought.git

%cd graph-of-thought

In [ ]:
# 2) Install training deps (Colab already ships a CUDA build of torch)
!pip install -q -r requirements.txt
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) Authenticate for the gated Llama-3.2 model (needs an HF_TOKEN Colab secret)
from google.colab import userdata
from huggingface_hub import login

login(userdata.get('HF_TOKEN'))  # Colab secret; never committed (AGENTS.md section 8)


## Phase 1 — hyperparameter tuning (merge heuristics × thresholds)

Sweeps every merge heuristic and threshold over 25 GSM8K questions and emits **one
dashboard per configuration** plus a comparison dashboard and `tuning_summary.json`.
Inspect the merge samples (highest-similarity vs borderline) to pick the heuristic +
threshold that merge semantically-equivalent states without collapsing incompatible ones.


In [ ]:
# Phase 1: run the tuning sweep, then preview the comparison dashboard inline
import glob, os, json
from IPython.display import HTML, display

get_ipython().system('python scripts/tune_graph.py --config configs/tune_gsm8k.json')

latest = sorted(glob.glob('output/*tune-gsm8k*/'))[-1].rstrip('/')
manifest = json.load(open(os.path.join(latest, 'run_manifest.json')))
print('Tuning run:', os.path.basename(latest), '| status:', manifest['status'])
if manifest['status'] != 'completed':
    # Interrupted (Ctrl+C) or failed runs have no outputs yet.
    print('Run did not complete. Errors:', manifest.get('errors'))
else:
    print('configs evaluated:', manifest['outputs']['configs_evaluated'])
    display(HTML(open(os.path.join(latest, manifest['outputs']['comparison_dashboard'])).read()))


## Phase 2 — full run with the selected heuristic + threshold

Set `graph.heuristic` and `graph.threshold` in `configs/graph_gsm8k.json` to the choice
from Phase 1, then run the full graph generation below.


In [ ]:
# Phase 2: generate reasoning graphs on GSM8K with the selected heuristic/threshold
!python scripts/generate_graphs.py --config configs/graph_gsm8k.json


In [ ]:
# 5) Zip + download the run (Colab disk is ephemeral) and preview the graph report inline
import glob, os, json
from google.colab import files
from IPython.display import HTML, display

latest = sorted(glob.glob('output/*graph-gsm8k*/'))[-1].rstrip('/')
run_id = os.path.basename(latest)
!zip -qr {run_id}.zip {latest}
files.download(f'{run_id}.zip')

manifest = json.load(open(os.path.join(latest, 'run_manifest.json')))
print('Run:', run_id, '| status:', manifest['status'])
if manifest['status'] != 'completed':
    print('Run did not complete. Errors:', manifest.get('errors'))
else:
    report = manifest['outputs']['reports'][0]
    print('graph report:', report)
    display(HTML(open(os.path.join(latest, report)).read()))


### Optional
- **CPU smoke test (no GPU, no token):** tuning `!python scripts/tune_graph.py --config configs/tune_demo.json` then full run `!python scripts/generate_graphs.py --config configs/graph_demo.json`
- **Validate a downloaded run locally:** `python scripts/validate_run.py output/<run_id>`
- **Sentiment fine-tuning demo (separate workload):** `!python scripts/train.py --config configs/train_sst2.json`
